In [0]:
%sql
-- ================================================================
-- SEED: table_config
-- ================================================================
TRUNCATE TABLE bfsi_lakehouse.metadata.table_config;
INSERT INTO bfsi_lakehouse.metadata.table_config (
    source_table_name, source_system, source_format, source_path_pattern,
    target_catalog, target_schema, target_table,
    partition_cols, replace_where_template, drift_policy,
    load_priority, is_active,
    created_at, created_by, last_edited_at, last_edited_by
)
VALUES
('t_Client', 'LMS_CORE', 'PARQUET',
 '/Volumes/bfsi_lakehouse/raw/source_synthetic_data/t_Client/dt={run_dt}/',
 'bfsi_lakehouse', 'bronze', 't_client',
 'dt', "dt = '{run_dt}'", 'EVOLVE',
 60, TRUE,
 current_timestamp(), 'rohan.m.mukherjee@gmail.com', current_timestamp(), 'rohan.m.mukherjee@gmail.com'),

('t_AccountCustomer', 'LMS_CORE', 'PARQUET',
 '/Volumes/bfsi_lakehouse/raw/source_synthetic_data/t_AccountCustomer/dt={run_dt}/',
 'bfsi_lakehouse', 'bronze', 't_accountcustomer',
 'dt', "dt = '{run_dt}'", 'EVOLVE',
 20, TRUE,
 current_timestamp(), 'rohan.m.mukherjee@gmail.com', current_timestamp(), 'rohan.m.mukherjee@gmail.com'),

('t_Loan', 'LMS_CORE', 'PARQUET',
 '/Volumes/bfsi_lakehouse/raw/source_synthetic_data/t_Loan/dt={run_dt}/',
 'bfsi_lakehouse', 'bronze', 't_loan',
 'dt', "dt = '{run_dt}'", 'EVOLVE',
 30, TRUE,
 current_timestamp(), 'rohan.m.mukherjee@gmail.com', current_timestamp(), 'rohan.m.mukherjee@gmail.com'),

('t_LoanInstallment', 'LMS_CORE', 'PARQUET',
 '/Volumes/bfsi_lakehouse/raw/source_synthetic_data/t_LoanInstallment/dt={run_dt}/',
 'bfsi_lakehouse', 'bronze', 't_loaninstallment',
 'dt', "dt = '{run_dt}'", 'EVOLVE',
 40, TRUE,
 current_timestamp(), 'rohan.m.mukherjee@gmail.com', current_timestamp(), 'rohan.m.mukherjee@gmail.com'),

('t_AccountTrx', 'LMS_CORE', 'PARQUET',
 '/Volumes/bfsi_lakehouse/raw/source_synthetic_data/t_AccountTrx/dt={run_dt}/',
 'bfsi_lakehouse', 'bronze', 't_accounttrx',
 'dt', "dt = '{run_dt}'", 'EVOLVE',
 50, TRUE,
 current_timestamp(), 'rohan.m.mukherjee@gmail.com', current_timestamp(), 'rohan.m.mukherjee@gmail.com');

In [0]:
%sql
-- ================================================================
-- SEED: table_process_config
-- ================================================================
TRUNCATE TABLE bfsi_lakehouse.metadata.table_process_config;
INSERT INTO bfsi_lakehouse.metadata.table_process_config (
    table_id, process_type, load_type,
    depends_on_table_ids, transform_module,
    is_active,
    created_at, created_by, last_edited_at, last_edited_by
)
SELECT
    tc.table_id,
    'BRONZE'              AS process_type,
    'INCREMENTAL'         AS load_type,
    CAST(NULL AS STRING)  AS depends_on_table_ids,  -- Bronze has no intra-layer deps
    CAST(NULL AS STRING)  AS transform_module,      -- Bronze ingests as-is
    TRUE                  AS is_active,
    current_timestamp(), 'rohan.m.mukherjee@gmail.com',
    current_timestamp(), 'rohan.m.mukherjee@gmail.com'
FROM bfsi_lakehouse.metadata.table_config tc
WHERE tc.source_table_name IN (
    't_Client', 't_AccountCustomer', 't_Loan', 't_LoanInstallment', 't_AccountTrx'
);

In [0]:
%sql
-- ================================================================
-- SEED: table_process_config — SILVER
-- ================================================================
INSERT INTO bfsi_lakehouse.metadata.table_process_config (
    table_id, process_type, load_type,
    depends_on_table_ids, transform_module,
    is_active,
    created_at, created_by, last_edited_at, last_edited_by
)
SELECT
    tc.table_id,
    'SILVER'              AS process_type,
    'INCREMENTAL'         AS load_type,
    deps.depends_on      AS depends_on_table_ids,
    deps.transform_mod   AS transform_module,
    TRUE                 AS is_active,
    current_timestamp(), 'rohan.m.mukherjee@gmail.com',
    current_timestamp(), 'rohan.m.mukherjee@gmail.com'
FROM bfsi_lakehouse.metadata.table_config tc
JOIN (
    VALUES
    ('t_Client',          CAST(NULL AS STRING),   'silver.transforms.t_client'),
    ('t_AccountCustomer', '1',                    'silver.transforms.t_account_customer'),
    ('t_Loan',            '1,2',                  'silver.transforms.t_loan'),
    ('t_LoanInstallment', '1,2,3',                'silver.transforms.t_loan_installment'),
    ('t_AccountTrx',      '1,2',                  'silver.transforms.t_account_trx')
) AS deps(table_name, depends_on, transform_mod)
ON tc.source_table_name = deps.table_name;

In [0]:
%sql
-- ================================================================
-- input_column_config — t_Client — SILVER
-- Replace :t_client_table_id with actual table_id from table_config
-- ================================================================

INSERT INTO bfsi_lakehouse.metadata.input_column_config
(table_id, process_type, source_column_name, target_column_name, data_type, is_nullable, is_pii, column_purpose, is_active, created_at, created_by, last_edited_at, last_edited_by)
VALUES
-- ── KEYS ──────────────────────────────────────────────────────────
(6, 'SILVER', 'ClientID',      NULL,       'STRING',    FALSE, FALSE, 'KEY',       TRUE, current_timestamp(), 'system', current_timestamp(), 'system'),
(6, 'SILVER', 'OurBranchID',   NULL,       'STRING',    FALSE, FALSE, 'KEY',       TRUE, current_timestamp(), 'system', current_timestamp(), 'system'),

-- ── ATTRIBUTES ────────────────────────────────────────────────────
(6, 'SILVER', 'ClientName',    NULL,       'STRING',    FALSE, TRUE,  'ATTRIBUTE', TRUE, current_timestamp(), 'system', current_timestamp(), 'system'),
(6, 'SILVER', 'ClientType',    NULL,       'STRING',    FALSE, FALSE, 'ATTRIBUTE', TRUE, current_timestamp(), 'system', current_timestamp(), 'system'),
(6, 'SILVER', 'DOB',           NULL,       'DATE',      TRUE,  TRUE,  'ATTRIBUTE', TRUE, current_timestamp(), 'system', current_timestamp(), 'system'),
(6, 'SILVER', 'IsActive',      NULL,       'BOOLEAN',   FALSE, FALSE, 'ATTRIBUTE', TRUE, current_timestamp(), 'system', current_timestamp(), 'system'),
(6, 'SILVER', 'State',         NULL,       'STRING',    TRUE,  FALSE, 'ATTRIBUTE', TRUE, current_timestamp(), 'system', current_timestamp(), 'system'),
(6, 'SILVER', 'Country',       NULL,       'STRING',    TRUE,  FALSE, 'ATTRIBUTE', TRUE, current_timestamp(), 'system', current_timestamp(), 'system'),

-- ── SOURCE AUDIT (pass-through from Bronze) ───────────────────────
(6, 'SILVER', 'CreatedAt',     NULL,       'TIMESTAMP', TRUE,  FALSE, 'AUDIT',     TRUE, current_timestamp(), 'system', current_timestamp(), 'system'),
(6, 'SILVER', 'CreatedBy',     NULL,       'STRING',    TRUE,  FALSE, 'AUDIT',     TRUE, current_timestamp(), 'system', current_timestamp(), 'system'),
(6, 'SILVER', 'LastUpdatedAt', NULL,       'TIMESTAMP', TRUE,  FALSE, 'AUDIT',     TRUE, current_timestamp(), 'system', current_timestamp(), 'system'),

-- ── DERIVED (Silver adds, not in Bronze) ──────────────────────────
(6, 'SILVER', NULL,  '_silver_processed_at', 'TIMESTAMP', FALSE, FALSE, 'DERIVED', TRUE, current_timestamp(), 'system', current_timestamp(), 'system');

In [0]:
%sql
SELECT * FROM bfsi_lakehouse.metadata.input_column_config;
SELECT * FROM bfsi_lakehouse.metadata.table_process_config WHERE process_type = 'SILVER';

SELECT * FROM bfsi_lakehouse.metadata.table_config;

UPDATE bfsi_lakehouse.metadata.table_process_config
SET depends_on_table_ids = CASE 
                                WHEN table_id = 6 THEN null 
                                WHEN table_id = 7 THEN '6'
                                WHEN table_id = 8 THEN  '6,7'
                                WHEN table_id = 9 THEN  '8'
                                WHEN table_id = 10 THEN '7,8'
                                ELSE depends_on_table_ids 
                            END
WHERE process_type = 'SILVER';

UPDATE bfsi_lakehouse.metadata.table_process_config
SET write_strategy = CASE 
                                WHEN table_id = 6 THEN 'SCD2' 
                                WHEN table_id = 7 THEN 'MERGE'
                                WHEN table_id = 8 THEN  'SCD2'
                                WHEN table_id = 9 THEN  'MERGE'
                                WHEN table_id = 10 THEN 'APPEND'
                                ELSE NULL 
                            END
WHERE process_type = 'SILVER'; 